# Hybrid DeBERTa + 2-Model Ensemble - All-in-One
- Trains DeBERTa with multiple seeds
- Identifies hard examples (top 10% by std across seeds)
- Runs 2-model ensemble (Qwen 32B + Llama 8B) inference only on hard examples
- Blends: 0.5 * deberta_prob + 0.5 * ensemble_prob for hard examples

## Part 1: DeBERTa Training

In [1]:
!uv pip install --system --no-index --find-links='/kaggle/input/jigsaw-packages2/whls/' 'trl==0.21.0' 'optimum==1.27.0' 'auto-gptq==0.7.1' 'bitsandbytes==0.46.1' 'deepspeed==0.17.4' 'logits-processor-zoo==0.2.1' 'vllm==0.10.0'
!uv pip install --system --no-index --find-links='/kaggle/input/jigsaw-packages2/whls/' 'triton==3.2.0'
!uv pip install --system --no-index --find-links='/kaggle/input/jigsaw-packages2/whls/' 'clean-text'
!uv pip install --system --no-index -U --no-deps --find-links='/kaggle/input/jigsaw-packages2/whls/' 'peft' 'accelerate' 'datasets'

Using Python 3.11.13 environment at: /usr
Resolved 168 packages in 560ms
   Building deepspeed==0.17.4
   Building deepspeed==0.17.4
   Building deepspeed==0.17.4
   Building deepspeed==0.17.4
   Building deepspeed==0.17.4
   Building deepspeed==0.17.4
   Building deepspeed==0.17.4
   Building deepspeed==0.17.4
   Building deepspeed==0.17.4
   Building deepspeed==0.17.4
   Building deepspeed==0.17.4
   Building deepspeed==0.17.4
   Building deepspeed==0.17.4
   Building deepspeed==0.17.4
   Building deepspeed==0.17.4
   Building deepspeed==0.17.4
   Building deepspeed==0.17.4
   Building deepspeed==0.17.4
   Building deepspeed==0.17.4
   Building deepspeed==0.17.4
   Building deepspeed==0.17.4
   Building deepspeed==0.17.4
   Building deepspeed==0.17.4
   Building deepspeed==0.17.4
   Building deepspeed==0.17.4
   Building deepspeed==0.17.4
   Building deepspeed==0.17.4
   Building deepspeed==0.17.4
   Building deepspeed==0.17.4
   Building deepspeed==0.17.4
   Building deepspeed==0.17

In [2]:
import os
import torch
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from torch import nn
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score
from transformers import AutoTokenizer, AutoModel, get_cosine_schedule_with_warmup
from tqdm import tqdm
import multiprocessing as mp
from transformers import get_linear_schedule_with_warmup
%env TOKENIZERS_PARALLELISM= 'false'

# %env KAGGLE_IS_COMPETITION_RERUN= 'true'

env: TOKENIZERS_PARALLELISM='false'


In [3]:
MAX_LEN = 256
BATCH_SIZE = 24
EPOCHS =  5
MODEL_PATH = "/kaggle/input/deberta-v3-base/transformers/default/1/deberta-v3-base"
SEEDS = [42, 123]
HARD_EXAMPLE_RATIO = 0.05

In [4]:
train_path = "/kaggle/input/jigsaw-agile-community-rules/train.csv"
test_path = "/kaggle/input/jigsaw-agile-community-rules/test.csv"
sample_sub_path = "/kaggle/input/jigsaw-agile-community-rules/sample_submission.csv"

In [5]:
df = pd.read_csv(train_path)
df["text"] = df["rule"] + " [SEP] " + df["body"]
df["label"] = df["rule_violation"].astype(float)

In [6]:
def add_data(dataframe):
    ret=[[],[]]
    for i in ['positive_example_1','positive_example_2','negative_example_1','negative_example_2']:
        tmp= (dataframe['rule']+' [SEP] '+ dataframe[i]).tolist()
        ret[0]+= tmp
        ret[1]+= [1]*len(tmp) if 'positive' in i else [0]*len(tmp)
    return ret

In [7]:
test_df = pd.read_csv(test_path)

augmented_train = add_data(df)
augmented_test = add_data(test_df)

augmented_texts =  df.text.tolist()+augmented_train[0] + augmented_test[0]
augmented_labels =  df.label.tolist()+augmented_train[1] + augmented_test[1]

augmented_df = pd.DataFrame({
    'text': augmented_texts,
    'label': augmented_labels
})
print(f'Before:{augmented_df.shape}')
augmented_df = augmented_df.groupby(augmented_df['text'].str.lower(), as_index=False).agg({
    'text': 'first',
    'label': 'mean'
})
print('After:',augmented_df.shape)
augmented_df['rule']= augmented_df.text.apply(lambda x: x.split(' [SEP] ')[0])
augmented_df['body']= augmented_df.text.apply(lambda x: x.split(' [SEP] ')[1])

rule_map= {i:j for j,i in enumerate(augmented_df.rule.str.lower().unique())}
augmented_df['rule_id']= augmented_df.rule.str.lower().map(rule_map)

augmented_df.head()

Before:(10185, 2)
After: (1875, 2)


,text,label,rule,body,rule_id
0,"No Advertising: Spam, referral links, unsolici...",1.0,"No Advertising: Spam, referral links, unsolici...","\n\nIf you have some free time on your hands, ...",0
1,"No Advertising: Spam, referral links, unsolici...",1.0,"No Advertising: Spam, referral links, unsolici...",\n\nplease visit http://www.shifadental.net/te...,0
2,"No Advertising: Spam, referral links, unsolici...",0.0,"No Advertising: Spam, referral links, unsolici...",\n\nSD | [ English Stream 1 Arsenal vs Tottenh...,0
3,"No Advertising: Spam, referral links, unsolici...",0.0,"No Advertising: Spam, referral links, unsolici...",\n**HD** ENG [ 1080P HD Amazing] :- [USTREAM E...,0
4,"No Advertising: Spam, referral links, unsolici...",1.0,"No Advertising: Spam, referral links, unsolici...",\nFree http://forums.airdroid.com/viewtopic.ph...,0


In [8]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH, use_fast = False)

In [9]:
if os.getenv('KAGGLE_IS_COMPETITION_RERUN'):
    for seed in SEEDS:
        train_data, val_data = train_test_split(
            augmented_df,
            test_size=0.2,
            stratify=augmented_df["rule"],
            random_state=seed
        )
        train_data.to_csv(f'fixed_train_split_seed_{seed}.csv', index=False)
        val_data.to_csv(f'fixed_val_split_seed_{seed}.csv', index=False)
        print(f'Seed {seed} splits saved: train={len(train_data)}, val={len(val_data)}')

In [10]:
class JigsawDataset(Dataset):
    def __init__(self, texts, labels,rule_ids, tokenizer, max_len):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len
        self.rule_ids = rule_ids

    def __len__(self): return len(self.texts)

    def __getitem__(self, idx):
        text = self.texts[idx]
        enc = self.tokenizer(
            text, padding='max_length', truncation=True, max_length=self.max_len, return_tensors="pt"
        )
        item = {k: v.squeeze(0) for k, v in enc.items()}
        item["labels"] = torch.tensor(self.labels[idx], dtype=torch.float)
        item['rule_ids']= torch.tensor(self.rule_ids[idx])
        return item

In [11]:
class JigsawModel(nn.Module):
    def __init__(self, model_path):
        super().__init__()
        self.base = AutoModel.from_pretrained(model_path)
        self.drop = nn.Dropout(0.15)
        self.out = nn.Linear(self.base.config.hidden_size, 1)

    def forward(self, input_ids, attention_mask):
        outputs = self.base(input_ids=input_ids, attention_mask=attention_mask)
        pooled = outputs.last_hidden_state[:, 0]
        return self.out(self.drop(pooled)).squeeze(1)

In [12]:
def train_one_epoch(model, loader, optimizer, scheduler, device):
    model.train()
    total_loss = 0
    for batch in tqdm(loader, desc='Training'):
        optimizer.zero_grad()
        input_ids = batch["input_ids"].to(device)
        mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)
        logits = model(input_ids, mask)
        loss = nn.BCEWithLogitsLoss()(logits, labels)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(),1.0)
        optimizer.step()
        if scheduler:
            scheduler.step()
        total_loss += loss.item()
    return total_loss / len(loader)

In [13]:
def validate(model, loader, device):
    model.eval()
    preds, targets, rule_ids_list = [], [], []
    total_loss = 0
    criterion = nn.BCEWithLogitsLoss()

    with torch.no_grad():
        for batch in loader:
            input_ids = batch["input_ids"].to(device)
            mask = batch["attention_mask"].to(device)
            labels = batch["labels"].to(device)
            rule_ids = batch["rule_ids"]

            logits = model(input_ids, mask)
            loss = criterion(logits, labels)
            total_loss += loss.item()

            preds.extend(torch.sigmoid(logits).cpu().numpy())
            targets.extend(labels.cpu().numpy())
            rule_ids_list.extend(rule_ids.cpu().numpy() if torch.is_tensor(rule_ids) else rule_ids)

    preds = np.array(preds)
    targets = np.array(targets)
    rule_ids_array = np.array(rule_ids_list)

    unique_rules = np.unique(rule_ids_array)
    rule_aucs = {}

    for rule_id in unique_rules:
        rule_mask = rule_ids_array == rule_id
        rule_preds = preds[rule_mask]
        rule_targets = targets[rule_mask]

        if len(np.unique(rule_targets >= 0.5)) > 1:
            rule_auc = roc_auc_score(rule_targets >= 0.5, rule_preds)
            rule_aucs[rule_id] = rule_auc
        else:
            rule_aucs[rule_id] = np.nan

    valid_aucs = [auc for auc in rule_aucs.values() if not np.isnan(auc)]
    avg_auc_per_rule = np.mean(valid_aucs) if valid_aucs else 0

    val_loss = total_loss / len(loader)
    return avg_auc_per_rule, val_loss, preds

In [14]:
def train_model_seed(seed, gpu_id):
    import random
    torch.manual_seed(seed)
    np.random.seed(seed)
    random.seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

    device = torch.device(f"cuda:{gpu_id}")
    print(f"[Seed {seed}] Training on {device}")

    train_data = pd.read_csv(f'fixed_train_split_seed_{seed}.csv')
    val_data = pd.read_csv(f'fixed_val_split_seed_{seed}.csv')

    train_ds = JigsawDataset(
        train_data['text'].tolist(),
        train_data['label'].tolist(),
        train_data['rule_id'].tolist(),
        tokenizer, MAX_LEN
    )

    val_ds = JigsawDataset(
        val_data['text'].tolist(),
        val_data['label'].tolist(),
        val_data['rule_id'].tolist(),
        tokenizer, MAX_LEN
    )

    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
    val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE)

    model = JigsawModel(MODEL_PATH).to(device)
    for name, param in model.named_parameters():
        if name.startswith('base.embedding'):
            param.requires_grad = False

    optimizer = torch.optim.AdamW(model.parameters(), lr=2e-5, eps=1e-6)
    total_steps = EPOCHS * len(train_loader)
    warmup_steps = int(0.1 * total_steps)
    scheduler = get_linear_schedule_with_warmup(
        optimizer,
        num_warmup_steps=warmup_steps,
        num_training_steps=total_steps,
    )

    best_auc = 0
    best_loss= None
    for epoch in range(EPOCHS):
        print(f"[Seed {seed}] Epoch {epoch+1}/{EPOCHS}")
        loss = train_one_epoch(model, train_loader, optimizer, scheduler, device)
        val_auc, val_loss, val_preds = validate(model, val_loader, device)

        print(f"[Seed {seed}] Loss: {loss:.4f}, Val Loss: {val_loss:.4f}, Val AUC: {val_auc:.4f}")
        if val_auc > best_auc:
            best_auc = val_auc
            best_loss= val_loss
            torch.save(model.state_dict(), f"model_seed_{seed}.bin")

    print(f"[Seed {seed}] Best validation AUC: {best_auc:.4f}")
    import json
    with open(f'results_seed_{seed}.json', 'w') as f:
        json.dump({'seed': seed, 'best_auc': best_auc, 'best_loss': best_loss}, f)

    return seed, best_auc

In [15]:
if os.getenv('KAGGLE_IS_COMPETITION_RERUN'):
    import torch.multiprocessing as mp
    mp.set_start_method('fork', force=True)

    processes = []
    for idx, seed in enumerate(SEEDS):
       gpu_id = idx % torch.cuda.device_count()
       p = mp.Process(target=train_model_seed, args=(seed, gpu_id))
       p.start()
       processes.append(p)

    for p in processes:
       p.join()

    import json
    results = []
    for seed in SEEDS:
      with open(f'results_seed_{seed}.json', 'r') as f:
          results.append(json.load(f))

    aucs = [r['best_auc'] for r in results]
    losses = [r['best_loss'] for r in results]

    print(f"AUC: {np.mean(aucs):.4f} ± {np.std(aucs):.4f}")
    print(f"Loss: {np.mean(losses):.4f} ± {np.std(losses):.4f}")

    print("All models trained!")

## Part 2: DeBERTa Inference & Identify Hard Examples

In [16]:
if os.getenv('KAGGLE_IS_COMPETITION_RERUN'):
    df_test = pd.read_csv(test_path)
    df_test["text"] = df_test["rule"] + " [SEP] " + df_test["body"]

    test_ds = JigsawDataset(df_test['text'].tolist(), [0]*len(df_test), [0]*len(df_test), tokenizer, MAX_LEN)
    test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE)

    all_preds = []

    for seed in SEEDS:
        device = torch.device("cuda:0")
        model = JigsawModel(MODEL_PATH).to(device)
        model.load_state_dict(torch.load(f"model_seed_{seed}.bin", map_location=device))
        model.eval()

        test_preds = []
        with torch.no_grad():
            for batch in tqdm(test_loader, desc=f"Inference seed {seed}"):
                ids = batch['input_ids'].to(device)
                mask = batch['attention_mask'].to(device)
                logits = model(ids, mask)
                test_preds.extend(torch.sigmoid(logits).cpu().numpy())

        all_preds.append(test_preds)
        
        seed_df = pd.DataFrame({
            'row_id': df_test['row_id'],
            f'pred_seed_{seed}': test_preds
        })
        seed_df.to_csv(f'test_preds_seed_{seed}.csv', index=False)
        print(f"Saved predictions for seed {seed}")

    seed_preds = np.array(all_preds)
    deberta_mean = seed_preds.mean(axis=0)
    deberta_std = seed_preds.std(axis=0)

    #print(f"\nMean predictions - min: {deberta_mean.min():.4f}, max: {deberta_mean.max():.4f}")
    #print(f"Std - min: {deberta_std.min():.4f}, max: {deberta_std.max():.4f}, mean: {deberta_std.mean():.4f}")

In [17]:
if os.getenv('KAGGLE_IS_COMPETITION_RERUN'):
    n_hard = int(len(deberta_mean) * HARD_EXAMPLE_RATIO)
    hard_indices = np.argsort(deberta_std)[-n_hard:]

    #print(f"Total examples: {len(deberta_mean)}")
    #print(f"Hard examples (top {HARD_EXAMPLE_RATIO*100}% by std): {n_hard}")
    #print(f"Std threshold: {deberta_std[hard_indices].min():.4f}")

    df_test['deberta_mean'] = deberta_mean
    df_test['deberta_std'] = deberta_std
    df_test['is_hard'] = False
    df_test.loc[hard_indices, 'is_hard'] = True

    #print(f"\nHard examples: {df_test['is_hard'].sum()}")

    df_hard = df_test[df_test['is_hard']].copy()
    #print(f"Hard examples for 14B inference: {len(df_hard)}")
    
    df_hard.to_csv('hard_examples.csv', index=False)
    #print("Saved hard examples to hard_examples.csv")

## Part 3: Write 2-Model Ensemble Inference Script & Run as Subprocess

In [18]:
%%writefile infer_qwen.py
#!/usr/bin/env python3
import os
import pandas as pd
from logits_processor_zoo.vllm import MultipleChoiceLogitsProcessor
import torch
import vllm
import numpy as np
from vllm.lora.request import LoRARequest
from scipy.special import softmax

df = pd.read_csv("hard_examples.csv")

MODEL_NAME = "/kaggle/input/qwen2-5-32b-instruct-gptq-int4"
LORA_PATH = "/kaggle/input/qwen2-5-32b-gptq-int4-batch4-full"

if __name__=='__main__':
  os.environ["VLLM_USE_V1"] = "0"

  llm = vllm.LLM(
      MODEL_NAME,
      quantization='gptq',
      tensor_parallel_size=torch.cuda.device_count(),
      gpu_memory_utilization=0.95,
      trust_remote_code=True,
      dtype="half",
      enforce_eager=True,
      max_model_len=4096,
      disable_log_stats=True,
      enable_prefix_caching=True,
      enable_lora=True,
  )
  tokenizer = llm.get_tokenizer()
  SYS_PROMPT = """
You are given a comment on reddit. Your task is to classify if it violates the given rule. Only respond Yes/No.
"""

  prompts = []
  for i, row in df.iterrows():
      text = f"""
r/{row.subreddit}
Rule: {row.rule}

1) {row.positive_example_1}
Violation: Yes

2) {row.positive_example_2}
Violation: Yes

3) {row.negative_example_1}
Violation: No

4) {row.negative_example_2}
Violation: No

5) {row.body}
"""

      messages = [
          {"role": "system", "content": SYS_PROMPT},
          {"role": "user", "content": text}
      ]

      prompt = tokenizer.apply_chat_template(
          messages,
          add_generation_prompt=True,
          tokenize=False,
      ) + "Answer:"
      prompts.append(prompt)

  mclp = MultipleChoiceLogitsProcessor(tokenizer, choices=['Yes','No'])
  outputs = llm.generate(
      prompts,
      vllm.SamplingParams(
          skip_special_tokens=True,
          max_tokens=1,
          logits_processors=[mclp],
          logprobs=2,
      ),
      use_tqdm=True,
      lora_request=LoRARequest("default", 1, LORA_PATH)
  )
  logprobs = [
      {lp.decoded_token: lp.logprob for lp in out.outputs[0].logprobs[0].values()}
      for out in outputs
  ]
  logit_matrix = pd.DataFrame(logprobs)[['Yes','No']]
  logit_matrix = logit_matrix.apply(lambda x: softmax(x.values), axis=1, result_type="expand")
  logit_matrix.columns = ['Yes', 'No']

  df['rule_violation'] = logit_matrix['Yes'].values
  df[['row_id', 'rule_violation']].to_csv("submission_qwen.csv",index=False)
  if torch.distributed.is_initialized():
      torch.distributed.destroy_process_group()

Writing infer_qwen.py


In [19]:
%%writefile infer_llama.py
#!/usr/bin/env python3
import os, math, numpy as np
os.environ["CUDA_VISIBLE_DEVICES"]="0,1"

import pandas as pd
import numpy as np

test = pd.read_csv('hard_examples.csv')

if __name__=='__main__':
    import vllm
    
    llm = vllm.LLM(
        "/kaggle/input/jigsaw-llama3-1-8b-instruct-training-one-epoch/llama-8b-instruct-jigsaw",
        tensor_parallel_size=2, 
        gpu_memory_utilization=0.95, 
        trust_remote_code=True,
        dtype="half", 
        enforce_eager=True,
        max_model_len=3000,
    )
    tokenizer = llm.get_tokenizer()
    
    
    from typing import Any, Dict, List
    from transformers import LogitsProcessor
    import torch
    
    choices = ["No", "Yes"]
    
    KEEP = []
    for x in choices:
        c = tokenizer.encode(x,add_special_tokens=False)[0]
        KEEP.append(c)
    print(f"Force predictions to be tokens {KEEP} which are {choices}.")
    
    class DigitLogitsProcessor(LogitsProcessor):
        def __init__(self, tokenizer):
            self.allowed_ids = KEEP
            
        def __call__(self, input_ids: List[int], scores: torch.Tensor) -> torch.Tensor:
            scores[self.allowed_ids] += 100
            return scores
    
    
    
    sys_prompt = '''You are given a comment on reddit and a rule. Your task is to classify whether the comment violates the rule. Only respond Yes/No.'''
    
    
    
    
    
    def formatting(dataset):
        texts = []
        for i in range(len(dataset)):
            texts.append(tokenizer.apply_chat_template(dataset[i], tokenize=False, add_generation_prompt=False))
        return texts
    
    
    
    
    
    template = """
Subreddit: r/{subreddit}
Rule: {rule}
Examples:
1) {positive_example_1}
Violation: Yes

2) {negative_example_1}
Violation: No

3) {negative_example_2}
Violation: No

4) {positive_example_2}
Violation: Yes
Comment:
{body}
Violation: """
    
    
    
    dataset = []
    for index,row in test.iterrows():
        
        formatted_sample = [
            {
            "role": "system",
            "content": sys_prompt
        },
           {
               "role": "user",
               "content": template.format(
                   rule = row.rule,
                   subreddit = row.subreddit,
                   body = row.body,
                   positive_example_1 = row.positive_example_1,
                   negative_example_1 = row.negative_example_1,
                   positive_example_2 = row.positive_example_2,
                   negative_example_2 = row.negative_example_2
               )
           }]
        
        dataset.append( formatted_sample )
    
    
    all_prompts = formatting(dataset)
    
    logits_processors = [DigitLogitsProcessor(tokenizer)]
    responses = llm.generate(
        all_prompts,
        vllm.SamplingParams(
            n=1,
            top_p=0.9,
            temperature=0,
            seed=777,
            skip_special_tokens=True,
            max_tokens=1,
            logits_processors=logits_processors,
            logprobs = 2
        ),
        use_tqdm = True
    )
    
    results = []
    errors = 0
    
    for i,response in enumerate(responses):
        try:
            x = response.outputs[0].logprobs[0]
            logprobs = []
            for k in KEEP:
                if k in x:
                    logprobs.append( math.exp(x[k].logprob) )
                else:
                    logprobs.append( 0 )
                    print(f"bad logits {i}")
            logprobs = np.array( logprobs )
            logprobs /= logprobs.sum()
            results.append( logprobs )
        except:
            results.append( np.array([1/2., 1/2.]) )
            errors += 1
            
    print(f"There were {errors} inference errors out of {i+1} inferences")
    results = np.vstack(results)
    
    probs = [x[1] for x in results]
    test['rule_violation'] = probs
    test[['row_id', 'rule_violation']].to_csv('submission_llama.csv',index=False)
    if torch.distributed.is_initialized():
        torch.distributed.destroy_process_group()

Writing infer_llama.py


In [20]:
if os.getenv('KAGGLE_IS_COMPETITION_RERUN'):
    import gc
    torch.cuda.empty_cache()
    gc.collect()
    print("GPU memory cleared before ensemble inference")

In [21]:
%%writefile ensemble_blend.py
#!/usr/bin/env python3
import pandas as pd
import numpy as np

q = pd.read_csv('submission_qwen.csv')
l = pd.read_csv('submission_llama.csv')

rq = q['rule_violation']#.rank(method='average') / (len(q)+1)
rl = l['rule_violation']#.rank(method='average') / (len(l)+1)

ensemble_prob = 0.6*rq + 0.4*rl

hard_examples = pd.read_csv('hard_examples.csv')
hard_examples['ensemble_prob'] = ensemble_prob.values

hard_examples.to_csv('hard_examples_with_ensemble.csv', index=False)
print(f"Ensemble complete! Saved to hard_examples_with_ensemble.csv")

Writing ensemble_blend.py


## Part 4: Run 2-Model Ensemble Inference as Sequential Subprocesses

In [22]:
if os.getenv('KAGGLE_IS_COMPETITION_RERUN'):
    print("Running Qwen 32B inference as subprocess...")
    !VLLM_USE_V1=0 python infer_qwen.py
    print("\nQwen 32B inference completed!")
    
    import gc
    torch.cuda.empty_cache()
    gc.collect()
    print("GPU memory cleared between models\n")
    
    print("Running Llama 8B inference as subprocess...")
    !python infer_llama.py
    print("\nLlama 8B inference completed!")
    
    print("\nCreating ensemble predictions...")
    !python ensemble_blend.py
    print("\nEnsemble blending completed!")

## Part 5: Blend & Submit

In [23]:
if os.getenv('KAGGLE_IS_COMPETITION_RERUN'):
    df_hard_with_ensemble = pd.read_csv('hard_examples_with_ensemble.csv')
    print(f"Loaded {len(df_hard_with_ensemble)} hard examples with ensemble predictions")
    print(f"\nEnsemble prob stats:")
    print(df_hard_with_ensemble['ensemble_prob'].describe())

Calibarate and then do an average?

In [24]:
if os.getenv('KAGGLE_IS_COMPETITION_RERUN'):
    df_hard_with_ensemble['blended_prob'] = 0.8 * df_hard_with_ensemble['deberta_mean'] + 0.2 * df_hard_with_ensemble['ensemble_prob']

    print(f"Blended {len(df_hard_with_ensemble)} hard examples")
    print(f"\nComparison (first 10):")
    print(df_hard_with_ensemble[['row_id', 'deberta_mean', 'deberta_std', 'ensemble_prob', 'blended_prob']].head(10))

In [25]:
if os.getenv('KAGGLE_IS_COMPETITION_RERUN'):
    df_test['final_pred'] = df_test['deberta_mean']

    for idx, row in df_hard_with_ensemble.iterrows():
        df_test.loc[idx, 'final_pred'] = float(row['blended_prob'])

In [26]:
if os.getenv('KAGGLE_IS_COMPETITION_RERUN'):
    submission = df_test[['row_id', 'final_pred']].copy()
    submission.columns = ['row_id', 'rule_violation']
    submission.to_csv('submission.csv', index=False)

    #print(f"Submission shape: {submission.shape}")
    #print(f"\nSubmission stats:")
    #print(submission['rule_violation'].describe())
else:
    !touch submission.csv

In [27]:
!head submission.csv

## Part 6: Analysis

In [28]:
if os.getenv('KAGGLE_IS_COMPETITION_RERUN'):
    df_hard_with_ensemble['diff'] = df_hard_with_ensemble['blended_prob'] - df_hard_with_ensemble['deberta_mean']